In [1]:
!pip install -q transformers datasets torch scikit-learn pandas tqdm

## Import Libraries and Packages

In [31]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset as HFDataset

import matplotlib.pyplot as plt
import seaborn as sns

## Import the Dataset

In [3]:
ds = load_dataset("AbstractTTS/IEMOCAP")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/489M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/456M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/462M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10039 [00:00<?, ? examples/s]

In [4]:
# Remove audio column
ds = ds.remove_columns(["audio"])

print(ds["train"].column_names)

['file', 'frustrated', 'angry', 'sad', 'disgust', 'excited', 'fear', 'neutral', 'surprise', 'happy', 'EmoAct', 'EmoVal', 'EmoDom', 'gender', 'transcription', 'major_emotion', 'speaking_rate', 'pitch_mean', 'pitch_std', 'rms', 'relative_db']


In [5]:
cols = ['transcription', 'major_emotion']

df = ds["train"].to_pandas()
df = df[cols]

In [6]:
df.head()

,transcription,major_emotion
0,Excuse me.,neutral
1,Yeah.,neutral
2,Is there a problem?,neutral
3,You did.,neutral
4,You were standing at the beginning and you di...,neutral


### Examine Missing Value

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10039 entries, 0 to 10038
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   transcription  10039 non-null  object
 1   major_emotion  10039 non-null  object
dtypes: object(2)
memory usage: 157.0+ KB


In [8]:
df.describe().T

,count,unique,top,freq
transcription,10039,8068,Yeah.,84
major_emotion,10039,10,frustrated,2917


In [9]:
print(f"\nTotal samples: {len(df)}")
print(f"\n{df.isnull().sum()}")

# Distribution of emotions
if 'major_emotion' in df.columns:
    print("\nEmotion Distribution:")
    print(df['major_emotion'].value_counts())


Total samples: 10039

transcription    0
major_emotion    0
dtype: int64

Emotion Distribution:
major_emotion
frustrated    2917
excited       1976
neutral       1726
angry         1269
sad           1250
happy          656
surprise       110
fear           107
other           26
disgust          2
Name: count, dtype: int64


In [10]:
# Checking for Empty strings
print(f"transcription : {len(df[df["transcription"] == " "])}")
print(f"major_emotion : {len(df[df["major_emotion"] == " "])}")

transcription : 0
major_emotion : 0


### Sample Transcriptions

In [11]:
for i, text in enumerate(df['transcription'].head(15), 1):
    print(f"{i:2d}. {text}")

 1.  Excuse me.
 2.  Yeah.
 3.  Is there a problem?
 4.  You did.
 5.  You were standing at the beginning and you directed me.
 6.  Well what's the problem?  Let me change it.
 7.  What?  I'm getting an ID.  This is why I'm here.  My wallet was stolen.
 8.  How am I supposed to get an ID without an ID?  How does a person get an ID in the first place?
 9.  I'm here to get an ID.
10.  Like what?  Like a birth certificate?
11.  Who the hell has a birth certificate?
12.  Yes but my wallet was stolen, I don't have anything.  I don't have any credit cards, I don't have my ID.  Don't you have things on file here?
13.  That's out of control.
14.  How long have you been working here?
15.  Clearly.  You know, do you have like a supervisor or something?


## Build the Confident Feature Extractor

In [12]:
class ConfidenceFeatureExtractor:
    """Extract linguistic features that indicate confidence levels"""
    
    def __init__(self):
        # Define linguistic markers
        self.hedging_words = [
            'maybe', 'perhaps', 'probably', 'might', 'could', 'possibly',
            'i think', 'i guess', 'kind of', 'sort of', 'somewhat'
        ]
        
        self.filler_words = [
            'um', 'uh', 'like', 'you know', 'well', 'actually', 'basically',
            'literally', 'so', 'anyway'
        ]
        
        self.confidence_words = [
            'definitely', 'certainly', 'absolutely', 'clearly', 'obviously',
            'sure', 'confident', 'positive', 'know', 'will'
        ]
        
        self.uncertainty_phrases = [
            'i don\'t know', 'not sure', 'uncertain', 'confused', 'unclear'
        ]
        
        self.assertive_words = [
            'must', 'will', 'shall', 'need to', 'have to', 'should'
        ]
    
    def extract_features(self, text):
        """Extract all confidence-related features from text"""
        text_lower = str(text).lower()
        words = text_lower.split()
        word_count = max(len(words), 1)  # Avoid division by zero
        
        features = {}
        
        # 1. Hedging words ratio
        features['hedging_ratio'] = sum(1 for hw in self.hedging_words if hw in text_lower) / word_count
        
        # 2. Filler words ratio
        features['filler_ratio'] = sum(1 for fw in self.filler_words if fw in text_lower) / word_count
        
        # 3. Confidence words ratio
        features['confidence_ratio'] = sum(1 for cw in self.confidence_words if cw in text_lower) / word_count
        
        # 4. Uncertainty phrases count
        features['uncertainty_count'] = sum(1 for up in self.uncertainty_phrases if up in text_lower)
        
        # 5. Assertive words ratio
        features['assertive_ratio'] = sum(1 for aw in self.assertive_words if aw in text_lower) / word_count
        
        # 6. Question marks (uncertainty indicator)
        features['question_ratio'] = text.count('?') / word_count
        
        # 7. Exclamation marks
        features['exclamation_ratio'] = text.count('!') / word_count
        
        # 8. Average word length
        features['avg_word_length'] = np.mean([len(word) for word in words]) if words else 0
        
        # 9. Sentence length
        features['sentence_length'] = word_count
        
        # 10. First person usage
        first_person = ['i ', 'me ', 'my ', 'mine ', 'myself ']
        features['first_person_ratio'] = sum(1 for fp in first_person if fp in text_lower + ' ') / word_count
        
        # 11. Politeness markers
        politeness = ['excuse me', 'please', 'sorry', 'pardon', 'thank you']
        features['politeness_ratio'] = sum(1 for pm in politeness if pm in text_lower) / word_count
        
        # 12. Direct statement
        features['is_statement'] = 1 if not text.strip().endswith('?') else 0
        
        # 13. Absolute words
        absolute_words = ['all', 'none', 'never', 'always', 'every', 'nothing']
        features['absolute_ratio'] = sum(1 for aw in absolute_words if aw in text_lower) / word_count
        
        return features
    
    def extract_batch(self, texts):
        """Extract features for multiple texts"""
        features_list = [self.extract_features(text) for text in texts]
        return pd.DataFrame(features_list)

### Test Feature Extraction

In [13]:
extractor = ConfidenceFeatureExtractor()

test_sentences = [
    "I definitely know the answer and will help you right away.",
    "Um, I think maybe we could possibly try to help with that.",
    "I don't know, perhaps someone else might be able to assist you."
]

for i, sentence in enumerate(test_sentences, 1):
    print(f"\nExample {i}: \"{sentence}\"")
    features = extractor.extract_features(sentence)
    print(f"  Confidence words: {features['confidence_ratio']:.3f}")
    print(f"  Hedging words: {features['hedging_ratio']:.3f}")
    print(f"  Filler words: {features['filler_ratio']:.3f}")
    print(f"  Assertiveness: {features['assertive_ratio']:.3f}")
    print(f"  Question?: {'Yes' if features['question_ratio'] > 0 else 'No'}")


Example 1: "I definitely know the answer and will help you right away."
  Confidence words: 0.273
  Hedging words: 0.000
  Filler words: 0.000
  Assertiveness: 0.091
  Question?: No

Example 2: "Um, I think maybe we could possibly try to help with that."
  Confidence words: 0.000
  Hedging words: 0.333
  Filler words: 0.083
  Assertiveness: 0.000
  Question?: No

Example 3: "I don't know, perhaps someone else might be able to assist you."
  Confidence words: 0.083
  Hedging words: 0.167
  Filler words: 0.083
  Assertiveness: 0.000
  Question?: No


## Build the Confident Detector

In [14]:
class ConfidenceDetector:
    """Main confidence detection model using unsupervised learning"""
    
    def __init__(self, n_clusters=3):
        self.n_clusters = n_clusters
        self.feature_extractor = ConfidenceFeatureExtractor()
        self.scaler = StandardScaler()
        self.kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        self.tfidf = TfidfVectorizer(max_features=100, ngram_range=(1, 2))
        self.cluster_labels = {}  # Maps cluster IDs to confidence levels
        self.feature_names = None
        
    def preprocess_data(self, df):
        """Clean and prepare the dataset"""
        df = df.copy()
        
        # Clean transcriptions
        df['transcription_clean'] = df['transcription'].apply(self._clean_text)
        
        return df
    
    def _clean_text(self, text):
        """Clean text data"""
        text = str(text).lower()
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    def fit(self, df):
        """Train the confidence detection model"""
        print("\nTraining Confidence Detector...")
        print("=" * 80)
        
        # Preprocess
        df = self.preprocess_data(df)
        print(f"  Dataset shape after preprocessing: {df.shape}")
        
        # Extract linguistic features
        print("  Extracting linguistic features...")
        linguistic_features = self.feature_extractor.extract_batch(df['transcription_clean'].tolist())
        self.feature_names = linguistic_features.columns.tolist()
        
        # Extract TF-IDF features
        print("  Extracting TF-IDF features...")
        tfidf_features = self.tfidf.fit_transform(df['transcription_clean']).toarray()
        
        # Combine features
        print("  Combining features...")
        combined_features = np.hstack([
            linguistic_features.values,
            tfidf_features
        ])
        
        # Scale features
        print("  Scaling features...")
        scaled_features = self.scaler.fit_transform(combined_features)
        
        # Cluster
        print(f"  Clustering into {self.n_clusters} confidence levels...")
        self.kmeans.fit(scaled_features)
        
        # Calculate silhouette score
        silhouette = silhouette_score(scaled_features, self.kmeans.labels_)
        print(f"  Silhouette Score: {silhouette:.3f} (higher is better)")
        
        # Assign cluster labels
        self._assign_confidence_labels(linguistic_features, self.kmeans.labels_)
        
        print("\n✅ Training complete!")
        return self
    
    def _assign_confidence_labels(self, linguistic_features, cluster_labels):
        """Assign cluster IDs to confidence levels based on characteristics"""
        cluster_stats = {}
        
        for cluster_id in range(self.n_clusters):
            mask = cluster_labels == cluster_id
            cluster_data = linguistic_features[mask]
            
            # Calculate confidence score
            confidence_score = (
                cluster_data['confidence_ratio'].mean() +
                cluster_data['assertive_ratio'].mean() +
                cluster_data['is_statement'].mean() -
                cluster_data['hedging_ratio'].mean() -
                cluster_data['filler_ratio'].mean() -
                cluster_data['question_ratio'].mean() -
                cluster_data['uncertainty_count'].mean()
            )
            
            cluster_stats[cluster_id] = confidence_score
        
        # Sort clusters by confidence score
        sorted_clusters = sorted(cluster_stats.items(), key=lambda x: x[1])
        
        # Assign labels
        self.cluster_labels = {
            sorted_clusters[0][0]: 'Low',
            sorted_clusters[1][0]: 'Medium' if self.n_clusters == 3 else 'Low-Medium',
        }
        
        if self.n_clusters == 3:
            self.cluster_labels[sorted_clusters[2][0]] = 'High'
        elif self.n_clusters >= 4:
            for i in range(2, self.n_clusters):
                if i == self.n_clusters - 1:
                    self.cluster_labels[sorted_clusters[i][0]] = 'High'
                else:
                    self.cluster_labels[sorted_clusters[i][0]] = f'Medium-{i-1}'
        
        print("\n  Cluster to Confidence mapping:")
        for cluster_id, label in self.cluster_labels.items():
            print(f"    Cluster {cluster_id} -> {label} (score: {cluster_stats[cluster_id]:.3f})")
    
    def predict(self, texts):
        """Predict confidence levels for new texts"""
        # Extract features
        linguistic_features = self.feature_extractor.extract_batch(texts)
        tfidf_features = self.tfidf.transform(texts).toarray()
        
        # Combine and scale
        combined_features = np.hstack([
            linguistic_features.values,
            tfidf_features
        ])
        scaled_features = self.scaler.transform(combined_features)
        
        # Predict clusters
        cluster_predictions = self.kmeans.predict(scaled_features)
        
        # Map to confidence labels
        confidence_predictions = [self.cluster_labels[cluster] for cluster in cluster_predictions]
        
        return confidence_predictions
    
    def predict_with_scores(self, texts):
        """Predict with detailed feature scores"""
        predictions = self.predict(texts)
        linguistic_features = self.feature_extractor.extract_batch(texts)
        
        results = pd.DataFrame({
            'transcription': texts,
            'confidence_level': predictions,
            'hedging_ratio': linguistic_features['hedging_ratio'],
            'filler_ratio': linguistic_features['filler_ratio'],
            'confidence_words': linguistic_features['confidence_ratio'],
            'assertiveness': linguistic_features['assertive_ratio']
        })
        
        return results

## Train the Model

In [15]:
detector = ConfidenceDetector(n_clusters=3)
detector.fit(df)


Training Confidence Detector...
  Dataset shape after preprocessing: (10039, 3)
  Extracting linguistic features...
  Extracting TF-IDF features...
  Combining features...
  Scaling features...
  Clustering into 3 confidence levels...
  Silhouette Score: 0.051 (higher is better)

  Cluster to Confidence mapping:
    Cluster 1 -> Low (score: -0.022)
    Cluster 0 -> Medium (score: 0.699)
    Cluster 2 -> High (score: 0.729)

✅ Training complete!


## Generate Predictions

In [16]:
results = detector.predict_with_scores(df['transcription'].tolist())
results.head(20)

,transcription,confidence_level,hedging_ratio,filler_ratio,confidence_words,assertiveness
0,Excuse me.,Medium,0.0,0.000000,0.000000,0.0
1,Yeah.,Medium,0.0,0.000000,0.000000,0.0
2,Is there a problem?,Medium,0.0,0.000000,0.000000,0.0
3,You did.,Medium,0.0,0.000000,0.000000,0.0
4,You were standing at the beginning and you di...,Medium,0.0,0.000000,0.000000,0.0
5,Well what's the problem? Let me change it.,Medium,0.0,0.125000,0.000000,0.0
6,What? I'm getting an ID. This is why I'm he...,Medium,0.0,0.000000,0.000000,0.0
7,How am I supposed to get an ID without an ID?...,Medium,0.0,0.045455,0.000000,0.0
8,I'm here to get an ID.,Medium,0.0,0.000000,0.000000,0.0
9,Like what? Like a birth certificate?,Medium,0.0,0.166667,0.000000,0.0


### Confident Level Distribuition

In [17]:
print("Confidence Level Distribution:")

conf_dist = results['confidence_level'].value_counts()
print("\nCounts:")
print(conf_dist)

print("\nPercentages:")
print((conf_dist / len(results) * 100).round(2))

Confidence Level Distribution:

Counts:
confidence_level
Medium    8863
High       872
Low        304
Name: count, dtype: int64

Percentages:
confidence_level
Medium    88.29
High       8.69
Low        3.03
Name: count, dtype: float64


### Example Transcriptions by Confidence Level

In [18]:
for level in ['Low', 'Medium', 'High']:
    mask = results['confidence_level'] == level
    if mask.sum() > 0:
        examples = results[mask]['transcription'].head(10)
        print(f"\n{level} Confidence ({mask.sum()} total samples):")
        for i, ex in enumerate(examples, 1):
            print(f"  {i:2d}. {ex}")


Low Confidence (304 total samples):
   1.  I don't know.  But I need an ID to pass this form along.  I can't just send it along without an ID.
   2.  I don't know.  I put in that. request too. They didn't...
   3.  I don't understand.  You've already done so much, I don't know why do you have to go back.
   4.  I don't know, it's just not fair.  I think you've already made your sacrifice.
   5.  I don't know how you can be okay with this?  I don't know why you're not-- you're fine with it.
   6.  I have an internet boyfriend.  I guess I'll have to juggle the two.  Is that cheating?  I don't know. Hmm.
   7.  I'm a big fan of the Wheel of Fortune quarters. But it just costs so much.  I don't know.
   8.  I don't know.  Isn't that-- that's all like escort services and things like that.
   9.  I don't know.  I mean I've been looking for so long that I just figured that--  I mean, I think things like Monster are a lot more reliable than just some random free board.
  10.  I don't know.  H

## NLP Model

In [28]:
label2id = {"Low": 0, "Medium": 1, "High": 2}
id2label = {v: k for k, v in label2id.items()}

results['confidence_id'] = results['confidence_level'].map(label2id)

train_df, test_df = train_test_split(
    results, test_size=0.25, stratify=results['confidence_id'], random_state=12
)

print(f"Train: {len(train_df)}   Test: {len(test_df)}")

Train: 7529   Test: 2510


In [36]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(
        examples["transcription"],
        padding="max_length",
        truncation=True,
        max_length=256                  # BERT handles longer context better
    )

train_dataset = HFDataset.from_pandas(train_df).map(tokenize_function, batched=True)
test_dataset  = HFDataset.from_pandas(test_df).map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column("confidence_id", "labels")
test_dataset  = test_dataset.rename_column("confidence_id", "labels")

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/7529 [00:00<?, ? examples/s]

Map:   0%|          | 0/2510 [00:00<?, ? examples/s]

In [37]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
training_args = TrainingArguments(
    output_dir="./confidence_bert_results",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = (predictions == labels).mean()
    return {"accuracy": acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss


In [39]:
eval_results = model.evaluate()
print(eval_results)

predictions = model.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print("\nClassification Report:")
print(classification_report(true_labels, pred_labels, target_names=["low", "medium", "high"]))

AttributeError: 'BertForSequenceClassification' object has no attribute 'evaluate'

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
)

examples = [
    "You are wasting my time! Fix this NOW!",
    "Thanks for the quick help!",
    "I don't know... maybe it's the cable?",
    "It's working now actually.",
    "Problem solved.",
    "Are you sure?",
    "Thanks a lot, It's working"
]

for ex in examples:
    result = classifier(ex, truncation=True, max_length=256)[0]
    print(f"\nText:\n{ex}\n→ {result['label']} (score: {result['score']:.3f})")